# 09 - Submission package

Package the notebooks, compact analysis results, report, and source metadata. Downloaded data, model checkpoints, and prediction partitions remain reproducible intermediates and are excluded.

**Inputs:** Notebooks 00-08 outputs and the compiled report.  
**Outputs:** `output/unicode_robust_text_classification_hwp.zip`.


## Setup

Run after compiling the report. Paths use the same configuration as the analysis notebooks.


In [1]:
from pathlib import Path
import zipfile
import yaml

# Same path setup as notebook 00; repeated for independent execution.
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
CONFIG = yaml.safe_load((ROOT / 'config/experiment.yaml').read_text())
PATHS = {key: ROOT / value for key, value in CONFIG['paths'].items()}
(ROOT / 'output').mkdir(exist_ok=True)


## Check and package deliverables

A compact presence check catches missing report inputs. Explicit file patterns exclude caches, checkpoints, and LaTeX build files without maintaining a separate manifest.


In [2]:
standalone = [ROOT / name for name in [
    'README.md', 'pyproject.toml', 'uv.lock', '.python-version', '.gitignore', 'output/pdf/main.pdf',
]] + [PATHS['metadata'], PATHS['logs'] / 'environment.json']
required = standalone + [PATHS['tables'] / name for name in [
    'metrics_summary.csv', 'principal_bootstrap_intervals.csv', 'paired_score_drop_bootstrap.csv',
    'attack_coverage.csv', 'token_failure_regression.csv',
]] + [PATHS['figures'] / 'robustness_curves.pdf', ROOT / 'report/main.tex', ROOT / 'report/references.bib']
missing = [str(path.relative_to(ROOT)) for path in required if not path.exists()]
assert not missing, f'Missing submission artifacts: {missing}'

files = list(standalone)
for directory, pattern in [
    (ROOT / 'notebooks', '[0-9][0-9]_*.ipynb'), (ROOT / 'config', '*.yaml'),
    (ROOT / 'report', '**/*.tex'), (ROOT / 'report', '*.bib'), (ROOT / 'report', '*.md'),
    (PATHS['tables'], '*.csv'), (PATHS['tables'], '*.parquet'), (PATHS['figures'], '*.pdf'),
]:
    files.extend(directory.glob(pattern))
archive = ROOT / 'output/unicode_robust_text_classification_hwp.zip'
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=9) as handle:
    for path in sorted(set(files)):
        handle.write(path, path.relative_to(ROOT))
{'archive': str(archive.relative_to(ROOT)), 'files': len(set(files)), 'size_mb': round(archive.stat().st_size / 1_000_000, 2)}


{'archive': 'output/unicode_robust_text_classification_hwp.zip',
 'files': 57,
 'size_mb': 2.11}